# 🚀 SSL400 — EXP5: EfficientNetV2-S + BiLSTM + CLAHE
## Standalone Notebook — Does NOT modify the main project files
---
**What is new in EXP5?**
- 🔄 **EfficientNetV2-S** replaces MobileNetV3Large as the backbone (more powerful!)
- 🔄 **BiLSTM** (Bidirectional LSTM) replaces standard LSTM (reads sequence both ways)
- ✅ **CLAHE + Gamma** preprocessing (same as EXP2, our best enhancement)
- ✅ **Reuses EXP2 processed .npy files** — no re-processing needed!

**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7**

In [ ]:
# Cell 1 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

In [ ]:
# Cell 2 — INSTALL DEPENDENCIES
!pip install tf-keras tf-models-official --quiet
print('✅ Dependencies ready!')

In [ ]:
# Cell 3 — CONFIGURATION
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# ===== SETTINGS — change these if needed =====
NUM_FRAMES    = 42          # Same as your main project
IMG_HEIGHT    = 224
IMG_WIDTH     = 224
NUM_CLASSES   = 8
BATCH_SIZE    = 2           # Keep at 2 for Tesla T4 GPU with EfficientNetV2
SEED          = 42

# EXP2 processed .npy files — REUSING because both use CLAHE!
PROCESSED_DIR = '/content/drive/MyDrive/SSL400_Research/data/processed/exp2_clahe_gamma'
SPLITS_DIR    = '/content/drive/MyDrive/SSL400_Research/data/splits'
TRAIN_CSV     = f'{SPLITS_DIR}/train_split.csv'
VAL_CSV       = f'{SPLITS_DIR}/val_split.csv'
TEST_CSV      = f'{SPLITS_DIR}/test_split.csv'

# Save locations — both Drive folders
LOCAL_MODEL_DIR  = '/content/exp5_model'
DRIVE_MODEL_DIR1 = '/content/drive/MyDrive/SSL400_Colab_Upload/models/experiment_5'
DRIVE_MODEL_DIR2 = '/content/drive/MyDrive/SSL400_Research/models/experiment_5'
LOG_PATH         = '/content/exp5_training_log.csv'

os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR1, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR2, exist_ok=True)

print(f'✅ Config ready! NUM_FRAMES={NUM_FRAMES}, BATCH_SIZE={BATCH_SIZE}')
print(f'📁 Using EXP2 CLAHE processed data from: {PROCESSED_DIR}')

In [ ]:
# Cell 4 — BUILD EfficientNetV2-S + BiLSTM MODEL
import tensorflow as tf
try:
    import tf_keras as keras
except ImportError:
    keras = tf.keras

import numpy as np
tf.random.set_seed(SEED)
np.random.seed(SEED)

def build_exp5_model(num_frames, img_h, img_w, num_classes):
    """EfficientNetV2-S + BiLSTM — EXP5 Architecture"""
    print('Building EfficientNetV2-S + BiLSTM model...')
    
    # 1. EfficientNetV2-S backbone (more powerful than MobileNetV3)
    backbone = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        pooling='avg',
        input_shape=(img_h, img_w, 3)
    )
    backbone.trainable = False  # Freeze for Phase 1
    
    # 2. Build full model
    inp = keras.Input(shape=(num_frames, img_h, img_w, 3), name='video_input')
    
    # Apply EfficientNetV2 to each frame independently
    x = keras.layers.TimeDistributed(backbone, name='efficientnetv2_td')(inp)
    
    # BatchNorm to stabilize after CNN
    x = keras.layers.TimeDistributed(keras.layers.BatchNormalization(), name='bn_td')(x)
    
    # BiLSTM — reads the frame sequence both FORWARD and BACKWARD
    x = keras.layers.Bidirectional(
        keras.layers.LSTM(256, return_sequences=False),
        name='bilstm'
    )(x)
    
    # Dropout to prevent overfitting
    x = keras.layers.Dropout(0.4, name='dropout')(x)
    
    # Final classification layer
    out = keras.layers.Dense(num_classes, activation='softmax', name='output')(x)
    
    model = keras.Model(inputs=inp, outputs=out)
    
    total = model.count_params()
    print(f'✅ Model built! Total parameters: {total:,} ({total/1e6:.1f}M)')
    print(f'   Backbone: EfficientNetV2-S')
    print(f'   Temporal: BiLSTM(256) — reads sequence both forward & backward')
    return model

model = build_exp5_model(NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES)
model.summary()

In [ ]:
# Cell 5 — BUILD DATASET PIPELINE
import csv
import glob

def load_npy_dataset(split_csv, processed_dir, num_frames, img_h, img_w, num_classes, batch_size):
    """Loads .npy files and returns a tf.data.Dataset."""
    
    # Read CSV split file
    samples = []
    with open(split_csv, 'r') as f:
        reader = csv.DictReader(f)
        label_col = 'label' if 'label' in reader.fieldnames else reader.fieldnames[-1]
        fname_col = reader.fieldnames[0]
        for row in reader:
            fname = row[fname_col]
            label = int(row[label_col])
            # Find the .npy file
            stem = os.path.splitext(fname)[0]
            npy_path = os.path.join(processed_dir, f"{stem}.npy")
            if os.path.exists(npy_path):
                samples.append((npy_path, label))

    print(f'  Found {len(samples)} .npy files for {os.path.basename(split_csv)}')
    
    if len(samples) == 0:
        raise ValueError(f"No .npy files found in {processed_dir}! Check your Drive path.")
    
    paths, labels = zip(*samples)
    paths  = list(paths)
    labels = list(labels)
    
    def generator():
        for path, label in zip(paths, labels):
            frames = np.load(path).astype(np.float32)
            # Resize frames if needed
            if frames.shape != (num_frames, img_h, img_w, 3):
                resized = []
                for i in range(min(num_frames, frames.shape[0])):
                    f = tf.image.resize(frames[i], (img_h, img_w)).numpy()
                    resized.append(f)
                while len(resized) < num_frames:
                    resized.append(resized[-1])
                frames = np.stack(resized[:num_frames])
            # EfficientNetV2 expects [0, 255] range (not normalized)
            frames = (frames * 127.5 + 127.5).clip(0, 255)  # un-normalize if needed
            one_hot = np.zeros(num_classes, dtype=np.float32)
            one_hot[label] = 1.0
            yield frames, one_hot
    
    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(num_frames, img_h, img_w, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(num_classes,), dtype=tf.float32)
        )
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(samples)

print('Loading datasets...')
train_ds, n_train = load_npy_dataset(TRAIN_CSV, PROCESSED_DIR, NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, BATCH_SIZE)
val_ds, n_val     = load_npy_dataset(VAL_CSV,   PROCESSED_DIR, NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, BATCH_SIZE)
test_ds, n_test   = load_npy_dataset(TEST_CSV,  PROCESSED_DIR, NUM_FRAMES, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, BATCH_SIZE)

print(f'\n✅ Datasets ready!')
print(f'   Train: {n_train} samples | Val: {n_val} samples | Test: {n_test} samples')

In [ ]:
# Cell 6 — TRAIN EXP5 (Phase 1: Frozen backbone, then Phase 2: Full fine-tune)
import shutil

def sync_to_drive(local_dir, drive_dir1, drive_dir2, log_path, phase):
    """Backup model + log to both Google Drive locations."""
    for drive_dir in [drive_dir1, drive_dir2]:
        try:
            os.makedirs(drive_dir, exist_ok=True)
            for f in os.listdir(local_dir):
                shutil.copy2(os.path.join(local_dir, f), os.path.join(drive_dir, f))
            if os.path.exists(log_path):
                shutil.copy2(log_path, os.path.join(drive_dir, f'training_log_{phase}.csv'))
        except Exception as e:
            print(f'  [Drive Sync Warning] {e}')
    print(f'  [Drive Sync] Backed up to both Drive locations!')

CHECKPOINT_P1 = os.path.join(LOCAL_MODEL_DIR, 'best_model_phase1.keras')
CHECKPOINT_P2 = os.path.join(LOCAL_MODEL_DIR, 'best_model_phase2.keras')

# ─────── PHASE 1: Frozen backbone — train only BiLSTM + head ───────
print('=' * 60)
print('  PHASE 1: Training with FROZEN EfficientNetV2-S backbone')
print('=' * 60)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_P1, monitor='val_loss', save_best_only=True, verbose=1),
    tf.keras.callbacks.CSVLogger(LOG_PATH, append=False),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

history_p1 = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks_p1)
sync_to_drive(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR1, DRIVE_MODEL_DIR2, LOG_PATH, 'phase1')

# ─────── PHASE 2: Full fine-tuning — unfreeze all layers ───────
print('\n' + '=' * 60)
print('  PHASE 2: Full fine-tuning (all layers unfrozen)')
print('=' * 60)

for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_P2, monitor='val_loss', save_best_only=True, verbose=1),
    tf.keras.callbacks.CSVLogger(LOG_PATH, append=False),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, verbose=1),
    tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: sync_to_drive(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR1, DRIVE_MODEL_DIR2, LOG_PATH, 'phase2') if (epoch+1) % 5 == 0 else None
    )
]

history_p2 = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks_p2)
sync_to_drive(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR1, DRIVE_MODEL_DIR2, LOG_PATH, 'phase2')
print('\n✅ Training complete! Model saved to both Google Drive locations.')

In [ ]:
# Cell 7 — EVALUATE EXP5 & SHOW PER-CLASS ACCURACY
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import json, time

CLASS_NAMES = ["Thank you", "Hello", "Good", "House", "Eat", "Drink", "Tell", "Write"]

print('Running evaluation on test set...')

# Get all predictions and true labels
all_logits = []
all_labels = []
for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    all_logits.append(preds)
    all_labels.append(y_batch.numpy())

logits  = np.concatenate(all_logits, axis=0)
y_true  = np.argmax(np.concatenate(all_labels, axis=0), axis=1)
y_pred  = np.argmax(logits, axis=1)

# Overall metrics
top1  = accuracy_score(y_true, y_pred)
f1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
prec  = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec   = recall_score(y_true, y_pred, average='macro', zero_division=0)

# Top-5 accuracy
top5_correct = sum(1 for true, logit in zip(y_true, logits) if true in np.argsort(logit)[-5:])
top5 = top5_correct / len(y_true)

# Per-class report
report_str  = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

print('\n' + '='*60)
print('  EXP5 RESULTS: EfficientNetV2-S + BiLSTM + CLAHE')
print('='*60)
print(f'  Top-1 Accuracy: {top1:.4f} ({top1*100:.2f}%)')
print(f'  Top-5 Accuracy: {top5:.4f} ({top5*100:.2f}%)')
print(f'  Macro F1:       {f1:.4f}')
print(f'  Macro Precision:{prec:.4f}')
print(f'  Macro Recall:   {rec:.4f}')
print(f'\n{report_str}')

# Compare vs EXP2
print('='*60)
print('  COMPARISON: EXP5 vs EXP2 (Best Previous)')
print('='*60)
print(f'  EXP2 (MobileNetV3 + LSTM + CLAHE):  71.43%')
print(f'  EXP5 (EfficientNetV2 + BiLSTM + CLAHE): {top1*100:.2f}%')
print(f'  Improvement: {(top1 - 0.7143)*100:+.2f} percentage points')

# Save results JSON
results = {
    'exp_id': 5,
    'exp_name': 'EfficientNetV2-S + BiLSTM + CLAHE',
    'top1_accuracy': float(top1),
    'top5_accuracy': float(top5),
    'macro_f1': float(f1),
    'macro_precision': float(prec),
    'macro_recall': float(rec),
    'classification_report': report_dict
}

res_path = '/content/exp5_metrics.json'
with open(res_path, 'w') as f:
    json.dump(results, f, indent=2)

# Backup results to Drive
import shutil
for drive_dir in [DRIVE_MODEL_DIR1, DRIVE_MODEL_DIR2]:
    shutil.copy2(res_path, os.path.join(drive_dir, 'exp5_metrics.json'))

print('\n✅ Results saved to Google Drive!')

In [ ]:
# Cell 8 — VISUALIZE: Training Curves + Per-Class Bar Chart
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ─── Training Curves ───
if os.path.exists(LOG_PATH):
    df = pd.read_csv(LOG_PATH)
    axes[0].plot(df['accuracy'], label='Train Acc', linewidth=2, color='#2ca02c')
    axes[0].plot(df['val_accuracy'], label='Val Acc', linewidth=2, color='#d62728', linestyle='--')
    axes[0].set_title('EXP5 Accuracy Curve', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].set_ylabel('Accuracy')
    axes[0].set_xlabel('Epoch')
    
    axes[1].plot(df['loss'], label='Train Loss', linewidth=2, color='#1f77b4')
    axes[1].plot(df['val_loss'], label='Val Loss', linewidth=2, color='#ff7f0e', linestyle='--')
    axes[1].set_title('EXP5 Loss Curve', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=12)
    axes[1].set_ylabel('Loss')
    axes[1].set_xlabel('Epoch')

# ─── Per-Class F1 Bar Chart ───
per_class_f1 = [report_dict[c]['f1-score'] * 100 for c in CLASS_NAMES]
colors = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 50 else '#e74c3c' for v in per_class_f1]
bars = axes[2].bar(CLASS_NAMES, per_class_f1, color=colors, edgecolor='black', linewidth=0.5)
axes[2].set_title('EXP5 Per-Class F1-Score', fontsize=14, fontweight='bold')
axes[2].set_ylabel('F1-Score (%)')
axes[2].set_ylim(0, 100)
axes[2].tick_params(axis='x', rotation=45)
axes[2].axhline(y=top1*100, color='blue', linestyle='--', linewidth=1.5, label=f'Overall ({top1*100:.1f}%)')
axes[2].legend()
for bar, val in zip(bars, per_class_f1):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/exp5_results.png', dpi=150, bbox_inches='tight')

# Backup figure to Drive
for drive_dir in [DRIVE_MODEL_DIR1, DRIVE_MODEL_DIR2]:
    shutil.copy2('/content/exp5_results.png', os.path.join(drive_dir, 'exp5_results.png'))

plt.show()
print('✅ Graph saved to Google Drive!')